## Prerequisite: SDEF Data

Run this ON YOUR MAC to generate the SDEF file (or use the existing one we created):



Then upload it in the cell below.


In [ ]:
from google.colab import files
import json

print("Click Browse to upload mac_app_sdefs.json from your Desktop")
uploaded = files.upload()

# Find the uploaded file
filename = list(uploaded.keys())[0]
APPS_SDEF_DATA = json.loads(uploaded[filename])
print(f"Loaded {len(APPS_SDEF_DATA)} apps with scripting dictionaries")

# Show what apps are available
for a in APPS_SDEF_DATA:
    cmd_count = a["sdef_xml"].count("command name=")
    print(f"  {a['name']}: ~{cmd_count} commands")


# MCP-x-Mac-Seed: Batch Evolution Notebook
---
**Feed macOS app SDEFs → LLM generates production AppleScript → outputs registrable MCP tool JSON**

This notebook batch-processes AppleScript dictionaries from macOS applications
and uses a free LLM (Gemma/Llama via Groq) to generate production-grade
AppleScript command mappings + MCP tool schemas.

**Output:** A JSON array of refined tools ready to import into MCP-x-Mac-Seed.

---
**Setup:** Runtime → Change runtime type → T4 GPU (free)

In [ ]:
# Install dependencies
!pip install -q groq pyyaml 2>/dev/null

import os, json, yaml, subprocess, sys, time, re, textwrap
from pathlib import Path

In [ ]:
# === CONFIGURATION ===

# 1. Get your Groq key from https://console.groq.com/keys
GROQ_API_KEY = "gsk_YOUR_KEY_HERE"  # ← PASTE YOUR GROQ KEY

# 2. Which model to use for AppleScript generation
LLM_MODEL = "llama-3.3-70b-versatile"  # Free on Groq, fast, great at code
# Alternatives: "mixtral-8x7b-32768", "gemma2-9b-it", "llama-3.1-8b-instant"

# 3. Which apps to evolve (empty = use all found)
TARGET_APPS = []  # e.g., ["Calendar", "Music", "Notes"]

# 4. Output file
OUTPUT_FILE = "evolved_tools.json"

print("Configured. Ready.")

## Step 1: Discover macOS App Locations

We need to find AppleScript-capable apps. This doesn't run `sdef` yet — it just
builds a list of app bundle paths to process.

In [ ]:
# Scan standard macOS app directories for .app bundles
APP_DIRS = [
    "/System/Applications",
    "/System/Applications/Utilities",
    "/Applications",
    "/Applications/Utilities",
]

apps = []
for d in APP_DIRS:
    if os.path.isdir(d):
        for item in sorted(os.listdir(d)):
            if item.endswith(".app") and not item.startswith("."):
                path = os.path.join(d, item)
                name = item[:-4]
                apps.append({"name": name, "path": path})

print(f"Found {len(apps)} app bundles")

# Filter to target apps if specified
if TARGET_APPS:
    apps = [a for a in apps if a["name"] in TARGET_APPS]
    print(f"Filtered to {len(apps)} target app(s): {[a['name'] for a in apps]}")

## Step 2: Extract SDEF Dictionaries

This runs the `sdef` command on each app and parses the XML into structured data.
On a free Colab GPU session, this takes ~30 seconds for 50 apps.

In [ ]:
import xml.etree.ElementTree as ET

def parse_app_commands(app):
    """Parse SDEF XML for a single app into structured command list."""
    commands = []
    try:
        root = ET.fromstring(app["sdef_xml"])
        for cmd in root.iter("{http://developer.apple.com/ns/sdef}command"):
            name = cmd.get("name", "")
            if cmd.get("hidden", "no") == "yes" or not name:
                continue
            parameters = []
            for param in cmd.findall("{http://developer.apple.com/ns/sdef}parameter"):
                parameters.append({
                    "name": param.get("name", ""),
                    "type": param.get("type", "any"),
                    "description": param.get("description", ""),
                    "optional": param.get("optional", "no") == "yes"
                })
            dp = cmd.find("{http://developer.apple.com/ns/sdef}direct-parameter")
            if dp is not None:
                parameters.insert(0, {
                    "name": "direct", "type": dp.get("type", "any"),
                    "description": dp.get("description", ""), "optional": False
                })
            commands.append({"name": name, "description": cmd.get("description", ""), "parameters": parameters})
    except:
        pass
    return commands

# Parse all apps
app_data = []
for app in APPS_SDEF_DATA:
    cmds = parse_app_commands(app)
    if cmds:
        app_data.append({"name": app["name"], "commands": cmds})
        print(f"  {app['name']}: {len(cmds)} commands")
    else:
        print(f"  {app['name']}: no parseable commands (skipped)")

print(f"
Total: {len(app_data)} apps, {sum(len(a['commands']) for a in app_data)} commands")


## Step 3: Generate AppleScript + MCP Tool Schemas via LLM

This is where the GPU comes in. We feed each app's SDEF commands to a free
LLM and ask it to generate:
1. Production-quality AppleScript for each command
2. A clean MCP tool schema with proper parameter types and descriptions

**This takes ~30-60 minutes for all 50 apps on free Groq tier.**

In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def generate_tool_for_command(app_name, command):
    """Use LLM to generate an MCP tool schema + AppleScript for one command."""
    
    prompt = f"""You are an AppleScript compiler. Given a macOS app name and a command from its scripting dictionary, output a JSON object with:
    1. "name": the MCP tool name ({{app_lowercase}}_{{command_name}})
    2. "description": clear description of what this tool does
    3. "app": the app name
    4. "strategy": "applescript"
    5. "isSensitive": true if the command modifies/deletes/sends data
    6. "appleScript": valid AppleScript code that executes the command
       - Parameters map to AppleScript variables
    7. "inputSchema": JSON Schema for the tool's parameters

App: {app_name}
Command: "{command['name']}"
Description: {command['description']}
Parameters:"""
    
    for p in command["parameters"]:
        prompt += f"\n  - {p['name']}: {p['type']} (optional: {p['optional']}) — {p['description']}"
    
    prompt += """

Examples of good output:
For App: Calendar, Command: "create event":
{{
  "name": "calendar_create_event",
  "description": "Create a new calendar event with title, date, duration, and optional notes",
  "app": "Calendar",
  "strategy": "applescript",
  "isSensitive": true,
  "appleScript": "tell application \"Calendar\"\\n	set newEvent to make new event at end of events with properties {{summary:\"{{summary}}\", start date:{{startDate}}, end date:{{startDate}} + ({{durationMinutes}} * minutes), description:\"{{notes}}\"}}\\n	return id of newEvent\\nend tell",
  "inputSchema": {{
    "type": "object",
    "properties": {{
      "summary": {{"type": "string", "description": "Event title"}},
      "startDate": {{"type": "string", "description": "Start date/time"}},
      "durationMinutes": {{"type": "number", "description": "Duration in minutes"}},
      "notes": {{"type": "string", "description": "Optional notes"}}
    }},
    "required": ["summary", "startDate"]
  }}
}}

Output ONLY valid JSON. No explanations, no markdown, just the JSON object."""
    
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "You are an AppleScript automation expert who outputs only JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=2000
        )
        
        text = response.choices[0].message.content.strip()
        
        # Strip markdown code fences if present
        if text.startswith("```"):
            text = text.split("\n", 1)[1]
            text = text.rsplit("```", 1)[0].strip()
        
        return json.loads(text)
    except Exception as e:
        print(f"  ⚠ LLM error for {app_name}/{command['name']}: {e}")
        return None

# Process apps in batch with progress tracking
all_tools = []
total = sum(len(a['commands']) for a in app_data)
completed = 0
errors = 0

print(f"Generating tools for {total} commands across {len(app_data)} apps...\n")
print("This will take a while. Let it run.")
print("="*50)

for app in app_data:
    for cmd in app["commands"]:
        tool = generate_tool_for_command(app["name"], cmd)
        completed += 1
        
        if tool:
            all_tools.append(tool)
            print(f"  ✅ [{completed}/{total}] {tool['name']}")
        else:
            errors += 1
            print(f"  ❌ [{completed}/{total}] {app['name']}_{cmd['name']}")
        
        # Rate limit: 30 req/min on Groq free
        if completed % 28 == 0:
            time.sleep(8)

print(f"\n{'='*50}")
print(f"Done: {len(all_tools)} tools generated, {errors} errors")

## Step 4: Export & Validate

The generated tools are saved as JSON. Each tool can be imported into MCP-x-Mac-Seed
via the `register_tool` MCP tool or SQLite directly.

In [ ]:
# Save to file
with open(OUTPUT_FILE, "w") as f:
    json.dump(all_tools, f, indent=2)

size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f"\nSaved: {OUTPUT_FILE} ({size_kb:.1f} KB)")
print(f"Tools: {len(all_tools)}")

# Show stats
apps_covered = len(set(t.get('app', '') for t in all_tools))
sensitive = sum(1 for t in all_tools if t.get('isSensitive'))
print(f"Apps covered: {apps_covered}")
print(f"Sensitive (gated): {sensitive}")
print()

# Preview first 3 tools
for t in all_tools[:3]:
    print(f"\n{'='*40}")
    print(f"Tool: {t.get('name')}")
    print(f"App: {t.get('app')}")
    print(f"Desc: {t.get('description')[:80]}...")
    print(f"Schema: {json.dumps(t.get('inputSchema', {}), indent=2)[:120]}...")

## Step 5: Download

Download the generated tools file and import it into your MCP-x-Mac-Seed instance.

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

print("\nDownload started. Import into MCP-x-Mac-Seed:")
print(f"1. Copy evolved_tools.json to your Mac")
print(f"2. Run: python3 import_evolved_tools.py evolved_tools.json")

## Import Script

Save the cell below as `import_evolved_tools.py` on your Mac and use it to register all
generated tools into the local MCP-x-Mac-Seed registry.

In [ ]:
IMPORT_SCRIPT = '''#!/usr/bin/env python3
"""Import evolved tools into MCP-x-Mac-Seed registry.
Usage: python3 import_evolved_tools.py evolved_tools.json
"""
import json, sqlite3, sys, os

DB_PATH = os.path.expanduser("~/Library/Application Support/MCPxMacSeed/tools.db")

def main():
    if len(sys.argv) < 2:
        print(f"Usage: {sys.argv[0]} evolved_tools.json")
        sys.exit(1)
    
    with open(sys.argv[1]) as f:
        tools = json.load(f)
    
    conn = sqlite3.connect(DB_PATH)
    registered = 0
    
    for tool in tools:
        name = tool.get("name", "")
        app = tool.get("app", "")
        schema = json.dumps({
            "name": name,
            "description": tool.get("description", ""),
            "app": app,
            "strategy": tool.get("strategy", "applescript"),
            "inputSchema": tool.get("inputSchema", {})
        }, indent=2)
        
        try:
            conn.execute("""
                INSERT INTO tools (name, app, version, schema_json, status, embedding, created_at, updated_at)
                VALUES (?, ?, 1, ?, 'active', NULL, datetime('now'), datetime('now'))
                ON CONFLICT(app, name) DO UPDATE SET
                    schema_json = excluded.schema_json,
                    version = version + 1,
                    status = 'active',
                    updated_at = datetime('now')
            """, (name, app, schema))
            registered += 1
        except Exception as e:
            print(f"  ❌ {name}: {e}")
    
    conn.commit()
    conn.close()
    print(f"Registered {registered}/{len(tools)} tools")

if __name__ == "__main__":
    main()
"""

with open("import_evolved_tools.py", "w") as f:
    f.write(IMPORT_SCRIPT)

print("Saved: import_evolved_tools.py")
files.download("import_evolved_tools.py")

## Troubleshooting

- **Groq rate limited (429):** Wait 1 minute, then re-run cell 3. Rate limits reset per minute.
- **SDEF empty for an app:** Not all apps have scripting dictionaries. Skipped automatically.
- **Colab disconnects:** Save progress. Re-run from cell 3, it skips already-processed tools.
- **Bad AppleScript:** The LLM generates code that may need manual tweaking. Each tool can be iteratively repaired via MCP-x-Mac-Seed's Repairman.

---
*Built for [MCP-x-Mac-Seed](https://github.com/reverendish/mcp-x-mac-seed) self-evolution pipeline*